###### Setup & Data Loading

In [2]:
import sys
import os
import pandas as pd
import numpy as np
import statsmodels.stats.api as sms

# Add src folder to module search path
sys.path.append(os.path.abspath("../src"))
from ab_testing import evaluate_ab_test

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Environment setup completed.")

Environment setup completed.


###### Formation of hypothesis

H_0:The new page has no effect on conversion rate

H_1: The new page produces a statistically significant difference

In [3]:
# Experiment Parameters
p_baseline = 0.120    
mde = 0.0035           
p_target = p_baseline + mde
alpha = 0.05          
power = 0.80         

# Calculate Cohen's h
cohens_h = 2 * (np.arcsin(np.sqrt(p_target)) - np.arcsin(np.sqrt(p_baseline)))

# Calculate required sample size per group
required_n = sms.NormalIndPower().solve_power(
    effect_size=cohens_h,
    power=power,
    alpha=alpha,
    ratio=1.0,
    alternative='two-sided'
)

required_n = int(np.ceil(required_n))

print(" Pre-Experiment Power Analysis ")
print(f"Baseline Conversion Rate (p1): {p_baseline:.1%}")
print(f"Target Conversion Rate (p2):   {p_target:.2%}")
print(f"Minimum Detectable Effect:    {mde:.2%}")
print(f"Effect Size (Cohen's h):       {cohens_h:.6f}")
print(f"Required Sample Size / Group:  {required_n:,} users")

 Pre-Experiment Power Analysis 
Baseline Conversion Rate (p1): 12.0%
Target Conversion Rate (p2):   12.35%
Minimum Detectable Effect:    0.35%
Effect Size (Cohen's h):       0.010704
Required Sample Size / Group:  137,015 users


###### Load Processed Data and Validate Sample Size

In [4]:
df_clean = pd.read_csv("../data/processed/ab_data_cleaned.csv")

# Group-level aggregation
summary = df_clean.groupby('group').agg(
    trials=('user_id', 'count'),
    conversions=('converted', 'sum'),
    conversion_rate=('converted', 'mean')
).reset_index()

print("Group Conversion Summary ")
display(summary)

control_trials = summary.loc[summary['group'] == 'control', 'trials'].values[0]
control_conv = summary.loc[summary['group'] == 'control', 'conversions'].values[0]

treatment_trials = summary.loc[summary['group'] == 'treatment', 'trials'].values[0]
treatment_conv = summary.loc[summary['group'] == 'treatment', 'conversions'].values[0]

# Verifying actual sample size against power requirement
actual_min_n = min(control_trials, treatment_trials)
is_sample_sufficient = actual_min_n >= required_n

print(f"\nMinimum Group Sample Size Collected: {actual_min_n:,}")
print(f"Sufficiently Powered (>= {required_n:,}): {is_sample_sufficient}")

Group Conversion Summary 


,group,trials,conversions,conversion_rate
0,control,145274,17489,0.120386
1,treatment,145310,17264,0.118808



Minimum Group Sample Size Collected: 145,274
Sufficiently Powered (>= 137,015): True


###### Executing Two-Proportion Z-Test & Computing 95% Confidence Interval

In [5]:
df_results = evaluate_ab_test(
    trials_a=control_trials,
    conversions_a=control_conv,
    trials_b=treatment_trials,
    conversions_b=treatment_conv,
    alpha=alpha
)

print(" Final A/B Test Statistical Evaluation")
display(df_results)

 Final A/B Test Statistical Evaluation


,Metric,Control (A),Treatment (B)
0,Sample Size,145274,145310
1,Conversions,17489,17264
2,Conversion Rate,12.0386%,11.8808%
3,Absolute Lift,-,-0.1578%
4,Relative Lift,-,-1.31%
5,Z-Statistic,-,-1.3109
6,p-value,-,0.1899
7,95% CI Lower,-,-0.3938%
8,95% CI Upper,-,+0.0781%
9,Statistically Significant?,-,False


###### Business Decision Framework & Export Statistical Report

In [6]:
# Extracting metrics for logging

p_val = float(df_results.loc[df_results['Metric'] == 'p-value', 'Treatment (B)'].values[0])
z_stat = float(df_results.loc[df_results['Metric'] == 'Z-Statistic', 'Treatment (B)'].values[0])
ci_lower = df_results.loc[df_results['Metric'] == '95% CI Lower', 'Treatment (B)'].values[0]
ci_upper = df_results.loc[df_results['Metric'] == '95% CI Upper', 'Treatment (B)'].values[0]

print("                           EXECUTIVE DECISION FRAMEWORK                         ")

print(f"• Null Hypothesis (H0):       p_treatment - p_control = 0")
print(f"• Significance Level (alpha):  {alpha}")
print(f"• Z-Statistic:                {z_stat:.4f}")
print(f"• p-value:                    {p_val:.4f}")
print(f"• 95% Confidence Interval:    [{ci_lower}, {ci_upper}]")

if p_val < alpha:
    decision = "REJECT NULL HYPOTHESIS (H0)"
    recommendation = "LAUNCH NEW PAGE: Statistically significant difference detected."
else:
    decision = "FAIL TO REJECT NULL HYPOTHESIS (H0)"
    recommendation = "DO NOT LAUNCH NEW PAGE: No statistically significant lift. Retain Control."

print(f"• Statistical Decision:       {decision}")
print(f"• Business Recommendation:    {recommendation}")

# Save summary report
os.makedirs("../reports", exist_ok=True)
df_results.to_csv("../reports/ab_test_statistical_summary.csv", index=False)
print("\nExported statistical summary to reports/ab_test_statistical_summary.csv")

                           EXECUTIVE DECISION FRAMEWORK                         
• Null Hypothesis (H0):       p_treatment - p_control = 0
• Significance Level (alpha):  0.05
• Z-Statistic:                -1.3109
• p-value:                    0.1899
• 95% Confidence Interval:    [-0.3938%, +0.0781%]
• Statistical Decision:       FAIL TO REJECT NULL HYPOTHESIS (H0)
• Business Recommendation:    DO NOT LAUNCH NEW PAGE: No statistically significant lift. Retain Control.

Exported statistical summary to reports/ab_test_statistical_summary.csv


In [7]:
# Adding Recommendation and Next Steps as rows to the results table
next_steps = (
    "Launch treatment to 100% of traffic."
    if p_val < alpha
    else "Keep the current version (Control). Consider testing for a longer period, "
         "trying a different variant, or increasing sample size."
)

extra_rows = pd.DataFrame([
    {"Metric": "Recommendation", "Control (A)": "-", "Treatment (B)": recommendation},
    {"Metric": "Next Steps", "Control (A)": "-", "Treatment (B)": next_steps}
])

df_results = pd.concat([df_results, extra_rows], ignore_index=True)

print("\nUpdated results table with Recommendation and Next Steps:")
display(df_results)

# Re-exporting with the new rows included
df_results.to_csv("../reports/ab_test_statistical_summary.csv", index=False)
print("\nRe-exported statistical summary to reports/ab_test_statistical_summary.csv")



Updated results table with Recommendation and Next Steps:


,Metric,Control (A),Treatment (B)
0,Sample Size,145274,145310
1,Conversions,17489,17264
2,Conversion Rate,12.0386%,11.8808%
3,Absolute Lift,-,-0.1578%
4,Relative Lift,-,-1.31%
5,Z-Statistic,-,-1.3109
6,p-value,-,0.1899
7,95% CI Lower,-,-0.3938%
8,95% CI Upper,-,+0.0781%
9,Statistically Significant?,-,False



Re-exported statistical summary to reports/ab_test_statistical_summary.csv
